# Decision policy: fine-tune the neural outcome model (the challenger to LightGBM)

The policy's trained challenger (D-096): an mmBERT encoder (Channel A's warm start) fine-tuned with two heads, P(unsafe) and P(fix), on the question, evidence, answer and the pre-action features written as text, for the controlled track's 13,383 fit records, with the 2,447 tune records for early stopping. It then scores the tune, calibration and test records; on a laptop, `faithguard policy --models lightgbm,neural` certifies it on the same grid as LightGBM. Attach the faithguard-outcome-data dataset (the rows `faithguard outcome-data` wrote from the laptop's cache, so both models see the same records; calibration and test rows carry no labels). Trains on one T4 in fp16 (D-098). Resumable. Expect about 1.5 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## The rows from the laptop (faithguard-outcome-data), or rebuilt here

In [ ]:
import glob
found = sorted(glob.glob('/kaggle/input/**/outcome-data.jsonl', recursive=True))
if found:
    DATA = found[0]
else:  # rebuilt from the downloaded datasets: slower, and TAT-QA items can differ slightly from the laptop's cache (D-088)
    !faithguard outcome-data
    DATA = 'runs/policy/neural/outcome-data.jsonl'
print(DATA)

## Fine-tune, then score the tune, calibration and test rows (resumable)

In [ ]:
!python -m faithguard.train.outcome_model --data $DATA --out {OUT}/outcome-model

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])